# 08 — Comparación de modelos en validación temporal 2024

Este notebook elige la **especificación candidata** usando únicamente 2024. Entrena con 2020–2023 y no lee ningún archivo de 2025. Compara el conjunto completo contra el parsimonioso de cada escenario, y modelos lineales y de árboles.

La selección no se basa en accuracy sola: se reportan macro-F1, balanced accuracy, precisión y recall de Severo, además de la matriz de confusión. Los pesos de clase son una alternativa transparente al sobre-muestreo para explorar el intercambio entre detección de clases minoritarias y falsas alertas; no se adopta ningún umbral operativo sin una decisión institucional explícita. Véanse la evidencia y decisiones previas en `registro_evidencia_metodologica.md` e `interpretacion_seleccion_caracteristicas.md`.

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'salidas_modelado' / 'bases_finales_riesgo'
OUTPUT_DIR = BASE_DIR / 'salidas_modelado' / 'comparacion_modelos_2024'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'NIVEL_DE_RIESGO_VICTIMA'
LABELS = ['1', '2', '3']
LABEL_NAMES = ['Leve', 'Moderado', 'Severo']
RANDOM_SEED = 20260827
MAX_TRAIN_ROWS = 200_000  # Igual que el baseline; usar None solo si se dispone de tiempo suficiente.
N_ARBOLES = 200
N_ITERACIONES_BOOSTING = 300
EJECUTAR_COMPARACION = True  # Cambiar a True para entrenar y validar.

FEATURES_PARSIMONIOSAS = {
    'inicial': [
        'VINCULO_AGRESOR_VICTIMA', 'PRIMERA_VEZ_AGREDE', 'ESTADO_AGRESOR_U_A', 'EDAD_VICTIMA',
        'NIVEL_EDUCATIVO_VICTIMA', 'AREA_RESIDENCIA_DOMICILIO', 'REDES_FAM_SOC', 'SEXO_AGRESOR', 'SEXO_VICTIMA',
    ],
    'retrospectiva': [
        'TIPO_VIOLENCIA', 'VINCULO_AGRESOR_VICTIMA', 'PRIMERA_VEZ_AGREDE', 'ESTADO_AGRESOR_U_A',
        'EDAD_VICTIMA', 'NIVEL_EDUCATIVO_VICTIMA', 'REDES_FAM_SOC', 'SEXO_AGRESOR', 'SEXO_VICTIMA',
    ],
}
print('Salida:', OUTPUT_DIR)

Salida: c:\Users\jcollazos\CODEX\salidas_modelado\comparacion_modelos_2024


In [9]:
def cargar_datos(escenario):
    rutas = {
        'train': DATA_DIR / f'riesgo_{escenario}_train_2020_2023.parquet',
        'validacion': DATA_DIR / f'riesgo_{escenario}_validacion_2024.parquet',
    }
    datos = {nombre: pd.read_parquet(ruta) for nombre, ruta in rutas.items()}
    for tabla in datos.values():
        assert tabla.isna().sum().sum() == 0, 'La base modelada no debe contener nulos.'
        tabla['EDAD_VICTIMA'] = pd.to_numeric(tabla['EDAD_VICTIMA'], errors='raise')
    return datos

def features_completas(tabla):
    excluir = {TARGET, 'nivel_riesgo_etiqueta', 'anio_ingreso'}
    return [col for col in tabla.columns if col not in excluir]

def muestra_estratificada(tabla):
    if MAX_TRAIN_ROWS is None or len(tabla) <= MAX_TRAIN_ROWS:
        return tabla.copy()
    muestra, _ = train_test_split(
        tabla, train_size=MAX_TRAIN_ROWS, stratify=tabla[TARGET], random_state=RANDOM_SEED
    )
    return muestra.copy()

def preprocesador_logistico(features):
    categoricas = [col for col in features if col != 'EDAD_VICTIMA']
    return ColumnTransformer([
        ('edad', StandardScaler(), ['EDAD_VICTIMA']),
        ('categorias', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categoricas),
    ])

def preprocesador_arboles(features):
    categoricas = [col for col in features if col != 'EDAD_VICTIMA']
    return ColumnTransformer([
        ('edad', 'passthrough', ['EDAD_VICTIMA']),
        ('categorias', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categoricas),
    ])

def preparar_catboost(tabla, features):
    resultado = tabla[features].copy()
    categoricas = [col for col in features if col != 'EDAD_VICTIMA']
    for col in categoricas:
        resultado[col] = resultado[col].astype(str)
    return resultado, categoricas

def construir_modelos(features):
    return {
        'logistica_sin_pesos': Pipeline([
            ('preprocesamiento', preprocesador_logistico(features)),
            ('modelo', LogisticRegression(max_iter=1000, solver='saga', n_jobs=-1, random_state=RANDOM_SEED)),
        ]),
        'logistica_balanceada': Pipeline([
            ('preprocesamiento', preprocesador_logistico(features)),
            ('modelo', LogisticRegression(max_iter=1000, solver='saga', class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED)),
        ]),
        'random_forest_balanceado': Pipeline([
            ('preprocesamiento', preprocesador_arboles(features)),
            ('modelo', RandomForestClassifier(
                n_estimators=N_ARBOLES, min_samples_leaf=5, max_features='sqrt',
                class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_SEED
            )),
        ]),
        'extra_trees_balanceado': Pipeline([
            ('preprocesamiento', preprocesador_arboles(features)),
            ('modelo', ExtraTreesClassifier(
                n_estimators=N_ARBOLES, min_samples_leaf=5, max_features='sqrt',
                class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED
            )),
        ]),
        'xgboost_multiclase': Pipeline([
            ('preprocesamiento', preprocesador_arboles(features)),
            ('modelo', XGBClassifier(
                objective='multi:softprob', num_class=3, n_estimators=N_ITERACIONES_BOOSTING,
                learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                eval_metric='mlogloss', tree_method='hist', n_jobs=-1, random_state=RANDOM_SEED
            )),
        ]),
        'catboost_balanceado': CatBoostClassifier(
            loss_function='MultiClass', iterations=N_ITERACIONES_BOOSTING, learning_rate=0.05, depth=6,
            auto_class_weights='Balanced', random_seed=RANDOM_SEED, thread_count=-1, verbose=False,
            allow_writing_files=False
        ),
    }

In [10]:
def metricas(y_real, y_predicho, escenario, conjunto, modelo):
    precision, recall, f1, soporte = precision_recall_fscore_support(
        y_real, y_predicho, labels=LABELS, zero_division=0
    )
    filas = [
        {'escenario': escenario, 'conjunto': conjunto, 'modelo': modelo, 'metrica': 'accuracy', 'clase': 'global', 'valor': accuracy_score(y_real, y_predicho)},
        {'escenario': escenario, 'conjunto': conjunto, 'modelo': modelo, 'metrica': 'balanced_accuracy', 'clase': 'global', 'valor': balanced_accuracy_score(y_real, y_predicho)},
        {'escenario': escenario, 'conjunto': conjunto, 'modelo': modelo, 'metrica': 'macro_f1', 'clase': 'global', 'valor': f1_score(y_real, y_predicho, labels=LABELS, average='macro', zero_division=0)},
    ]
    for etiqueta, p, r, f, s in zip(LABEL_NAMES, precision, recall, f1, soporte):
        filas.extend([
            {'escenario': escenario, 'conjunto': conjunto, 'modelo': modelo, 'metrica': 'precision', 'clase': etiqueta, 'valor': p},
            {'escenario': escenario, 'conjunto': conjunto, 'modelo': modelo, 'metrica': 'recall', 'clase': etiqueta, 'valor': r},
            {'escenario': escenario, 'conjunto': conjunto, 'modelo': modelo, 'metrica': 'f1', 'clase': etiqueta, 'valor': f},
            {'escenario': escenario, 'conjunto': conjunto, 'modelo': modelo, 'metrica': 'soporte', 'clase': etiqueta, 'valor': s},
        ])
    return filas

if EJECUTAR_COMPARACION:
    resultados = []
    for escenario in ['inicial', 'retrospectiva']:
        datos = cargar_datos(escenario)
        configuraciones = {
            'completo': features_completas(datos['train']),
            'parsimonioso': FEATURES_PARSIMONIOSAS[escenario],
        }
        train = muestra_estratificada(datos['train'])
        for nombre_conjunto, features in configuraciones.items():
            for nombre_modelo, modelo in construir_modelos(features).items():
                print(f'Ajustando: {escenario} | {nombre_conjunto} | {nombre_modelo}')
                y_real = datos['validacion'][TARGET].astype('string')
                if nombre_modelo == 'catboost_balanceado':
                    X_train_cat, categoricas_cat = preparar_catboost(train, features)
                    X_validacion_cat, _ = preparar_catboost(datos['validacion'], features)
                    modelo.fit(X_train_cat, train[TARGET].astype(int), cat_features=categoricas_cat)
                    y_predicho = np.asarray(modelo.predict(X_validacion_cat)).reshape(-1).astype(str)
                elif nombre_modelo == 'xgboost_multiclase':
                    modelo.fit(train[features], train[TARGET].astype(int) - 1)
                    y_predicho = (modelo.predict(datos['validacion'][features]).astype(int) + 1).astype(str)
                else:
                    modelo.fit(train[features], train[TARGET].astype('string'))
                    y_predicho = modelo.predict(datos['validacion'][features])
                resultados.extend(metricas(y_real, y_predicho, escenario, nombre_conjunto, nombre_modelo))
                matriz = confusion_matrix(y_real, y_predicho, labels=LABELS)
                matriz_df = pd.DataFrame(matriz, index=LABEL_NAMES, columns=LABEL_NAMES)
                matriz_df.index.name, matriz_df.columns.name = 'real', 'predicho'
                matriz_df.to_csv(OUTPUT_DIR / f'matriz_2024_{escenario}_{nombre_conjunto}_{nombre_modelo}.csv', encoding='utf-8-sig')
    resultados_df = pd.DataFrame(resultados)
    resultados_df.to_csv(OUTPUT_DIR / 'metricas_comparacion_validacion_2024.csv', index=False, encoding='utf-8-sig')
    resumen = resultados_df.query("metrica in ['macro_f1', 'balanced_accuracy']").pivot(
        index=['escenario', 'conjunto', 'modelo'], columns='metrica', values='valor'
    ).sort_values(['macro_f1', 'balanced_accuracy'], ascending=False)
    display(resumen)
else:
    print('Comparación no ejecutada. Cambie EJECUTAR_COMPARACION a True para entrenar.')

Ajustando: inicial | completo | logistica_sin_pesos


C:\Users\jcollazos\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Ajustando: inicial | completo | logistica_balanceada


C:\Users\jcollazos\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Ajustando: inicial | completo | random_forest_balanceado
Ajustando: inicial | completo | extra_trees_balanceado
Ajustando: inicial | completo | xgboost_multiclase
Ajustando: inicial | completo | catboost_balanceado
Ajustando: inicial | parsimonioso | logistica_sin_pesos


...

C:\Users\jcollazos\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Ajustando: retrospectiva | parsimonioso | logistica_balanceada


C:\Users\jcollazos\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Ajustando: retrospectiva | parsimonioso | random_forest_balanceado
Ajustando: retrospectiva | parsimonioso | extra_trees_balanceado
Ajustando: retrospectiva | parsimonioso | xgboost_multiclase
Ajustando: retrospectiva | parsimonioso | catboost_balanceado


metrica                                              balanced_accuracy  \
escenario     conjunto     modelo                                        
retrospectiva completo     random_forest_balanceado           0.502915   
                           extra_trees_balanceado             0.503906   
                           logistica_balanceada               0.505041   
                           catboost_balanceado                0.512398   
inicial       parsimonioso catboost_balanceado                0.452055   
              completo     catboost_balanceado                0.456251   
retrospectiva parsimonioso random_forest_balanceado           0.499337   
inicial       completo     random_forest_balanceado           0.450740   
retrospectiva parsimonioso extra_trees_balanceado             0.502403   
inicial       parsimonioso random_forest_balanceado           0.442453   
                           extra_trees_balanceado             0.445513   
              completo     extra_trees_balanceado             0.449282   
retrospectiva parsimonioso catboost_balanceado                0.508150   
              completo     xgboost_multiclase                 0.421456   
              parsimonioso logistica_balanceada               0.503702   
inicial       completo     logistica_balanceada               0.443227   
              parsimonioso logistica_balanceada               0.437676   
retrospectiva parsimonioso xgboost_multiclase                 0.409210   
              completo     logistica_sin_pesos                0.406390   
              parsimonioso logistica_sin_pesos                0.397480   
inicial       completo     xgboost_multiclase                 0.386184   
              parsimonioso xgboost_multiclase                 0.382643   
              completo     logistica_sin_pesos                0.364882   
              parsimonioso logistica_sin_pesos                0.363801   

metrica                                              macro_f1  
escenario     conjunto     modelo                              
retrospectiva completo     random_forest_balanceado  0.439215  
                           extra_trees_balanceado    0.431091  
                           logistica_balanceada      0.426118  
                           catboost_balanceado       0.425683  
inicial       parsimonioso catboost_balanceado       0.424682  
              completo     catboost_balanceado       0.422614  
retrospectiva parsimonioso random_forest_balanceado  0.420477  
inicial       completo     random_forest_balanceado  0.418183  
retrospectiva parsimonioso extra_trees_balanceado    0.415806  
inicial       parsimonioso random_forest_balanceado  0.415270  
                           extra_trees_balanceado    0.413195  
              completo     extra_trees_balanceado    0.412165  
retrospectiva parsimonioso catboost_balanceado       0.410779  
              completo     xgboost_multiclase        0.409818  
              parsimonioso logistica_balanceada      0.409384  
inicial       completo     logistica_balanceada      0.408900  
              parsimonioso logistica_balanceada      0.405496  
retrospectiva parsimonioso xgboost_multiclase        0.388878  
              completo     logistica_sin_pesos       0.385817  
              parsimonioso logistica_sin_pesos       0.368286  
inicial       completo     xgboost_multiclase        0.353254  
              parsimonioso xgboost_multiclase        0.347068  
              completo     logistica_sin_pesos       0.309912  
              parsimonioso logistica_sin_pesos       0.306387

In [11]:
# Lectura de resultados guardados: la decisión requiere métricas globales Y clase Severo.
ruta_metricas = OUTPUT_DIR / 'metricas_comparacion_validacion_2024.csv'
if ruta_metricas.exists():
    metricas_guardadas = pd.read_csv(ruta_metricas)
    resumen_global = metricas_guardadas.query("metrica in ['macro_f1', 'balanced_accuracy', 'accuracy']").pivot(
        index=['escenario', 'conjunto', 'modelo'], columns='metrica', values='valor'
    )
    resumen_severo = metricas_guardadas.query("clase == 'Severo' and metrica in ['precision', 'recall', 'f1']").pivot(
        index=['escenario', 'conjunto', 'modelo'], columns='metrica', values='valor'
    )
    display(resumen_global.join(resumen_severo, rsuffix='_severo').sort_values(['macro_f1', 'recall'], ascending=False))
else:
    print('Aún no existe el archivo de métricas. Ejecute primero la comparación.')

metrica                                              accuracy  \
escenario     conjunto     modelo                               
retrospectiva completo     random_forest_balanceado  0.436644   
                           extra_trees_balanceado    0.429492   
                           logistica_balanceada      0.424809   
                           catboost_balanceado       0.426359   
inicial       parsimonioso catboost_balanceado       0.431231   
              completo     catboost_balanceado       0.426471   
retrospectiva parsimonioso random_forest_balanceado  0.419652   
inicial       completo     random_forest_balanceado  0.422180   
retrospectiva parsimonioso extra_trees_balanceado    0.416305   
inicial       parsimonioso random_forest_balanceado  0.421705   
                           extra_trees_balanceado    0.417800   
              completo     extra_trees_balanceado    0.414862   
retrospectiva parsimonioso catboost_balanceado       0.412666   
              completo     xgboost_multiclase        0.546228   
              parsimonioso logistica_balanceada      0.410364   
inicial       completo     logistica_balanceada      0.412963   
              parsimonioso logistica_balanceada      0.410233   
retrospectiva parsimonioso xgboost_multiclase        0.538945   
              completo     logistica_sin_pesos       0.541023   
              parsimonioso logistica_sin_pesos       0.536251   
inicial       completo     xgboost_multiclase        0.528393   
              parsimonioso xgboost_multiclase        0.526844   
              completo     logistica_sin_pesos       0.522879   
              parsimonioso logistica_sin_pesos       0.523313   

metrica                                              balanced_accuracy  \
escenario     conjunto     modelo                                        
retrospectiva completo     random_forest_balanceado           0.502915   
                           extra_trees_balanceado             0.503906   
                           logistica_balanceada               0.505041   
                           catboost_balanceado                0.512398   
inicial       parsimonioso catboost_balanceado                0.452055   
              completo     catboost_balanceado                0.456251   
retrospectiva parsimonioso random_forest_balanceado           0.499337   
inicial       completo     random_forest_balanceado           0.450740   
retrospectiva parsimonioso extra_trees_balanceado             0.502403   
inicial       parsimonioso random_forest_balanceado           0.442453   
                           extra_trees_balanceado             0.445513   
              completo     extra_trees_balanceado             0.449282   
retrospectiva parsimonioso catboost_balanceado                0.508150   
              completo     xgboost_multiclase                 0.421456   
              parsimonioso logistica_balanceada               0.503702   
inicial       completo     logistica_balanceada               0.443227   
              parsimonioso logistica_balanceada               0.437676   
retrospectiva parsimonioso xgboost_multiclase                 0.409210   
              completo     logistica_sin_pesos                0.406390   
              parsimonioso logistica_sin_pesos                0.397480   
inicial       completo     xgboost_multiclase                 0.386184   
              parsimonioso xgboost_multiclase                 0.382643   
              completo     logistica_sin_pesos                0.364882   
              parsimonioso logistica_sin_pesos                0.363801   

metrica                                              macro_f1        f1  \
escenario     conjunto     modelo                                         
retrospectiva completo     random_forest_balanceado  0.439215  0.506618   
                           extra_trees_balanceado    0.431091  0.511864   
                           logistica_balanceada      0.426118  0.509021   
         

## Regla de decisión después de ejecutar

Se elegirá en 2024 la configuración que logre el mejor equilibrio entre macro-F1 y balanced accuracy, sin aceptar una ganancia global que reduzca materialmente el recall de Severo. No se fija todavía un mínimo numérico de recall: ese umbral representa una decisión de riesgo, capacidad de atención y costo de falsas alertas que debe justificarse con el contexto institucional, no deducirse automáticamente de los datos.

Tras interpretar la tabla, se congelará una sola especificación. Solo esa especificación pasará a una evaluación confirmatoria en 2025.